In [1]:
!pip -q install kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download(
    "amananandrai/ag-news-classification-dataset"
)

print(path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
/kaggle/input/ag-news-classification-dataset


In [3]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/ag-news-classification-dataset/train.csv
/kaggle/input/ag-news-classification-dataset/test.csv


In [4]:
import pandas as pd

train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

print(train_df.shape)
print(test_df.shape)

train_df.head()

(120000, 3)
(7600, 3)


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [5]:
print(train_df.columns)

Index(['Class Index', 'Title', 'Description'], dtype='object')


## Make one text column

In [6]:
train_df["text"] = (
    train_df["Title"].astype(str)
    + " "
    + train_df["Description"].astype(str)
)

test_df["text"] = (
    test_df["Title"].astype(str)
    + " "
    + test_df["Description"].astype(str)
)

## Tokenization

In [7]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

In [8]:
tokenize("Microsoft announces NEW AI products!")

['microsoft', 'announces', 'new', 'ai', 'products']

## Build vocabulary

In [9]:
from collections import Counter

counter = Counter()

for text in train_df["text"]:
    counter.update(tokenize(text))

In [10]:
counter.most_common(20)

[('the', 205468),
 ('to', 120734),
 ('a', 113341),
 ('of', 98647),
 ('in', 96424),
 ('and', 69668),
 ('s', 61984),
 ('on', 57660),
 ('for', 50674),
 ('39', 44505),
 ('that', 28168),
 ('with', 26809),
 ('as', 25376),
 ('at', 25068),
 ('its', 22123),
 ('is', 22097),
 ('new', 21424),
 ('by', 20931),
 ('it', 20538),
 ('said', 20267)]

In [11]:
MAX_VOCAB = 30000

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in counter.most_common(MAX_VOCAB - 2):
    word2idx[word] = len(word2idx)

VOCAB_SIZE = len(word2idx)

print("Vocabulary:", VOCAB_SIZE)

Vocabulary: 30000


`<PAD>`

Used to make sequences equal length.

`<UNK>`

Unknown word.

## Encoder

In [12]:
MAX_LEN = 80

def encode_text(text):
    tokens = tokenize(text)

    ids = [
        word2idx.get(token, word2idx["<UNK>"])
        for token in tokens
    ]

    ids = ids[:MAX_LEN]

    length = len(ids)

    ids += [word2idx["<PAD>"]] * (MAX_LEN - len(ids))

    return ids, length

## Dataset

In [13]:
from torch.utils.data import Dataset

class NewsDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.df.loc[idx, "text"]

        ids, length = encode_text(text)

        label = int(self.df.loc[idx, "Class Index"]) - 1

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

## Train/validation split

In [14]:

from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["Class Index"]
)

## DataLoader

In [15]:
from torch.utils.data import DataLoader

train_dataset = NewsDataset(train_part)
val_dataset = NewsDataset(val_part)
test_dataset = NewsDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

## Build the RNN

In [16]:
import torch.nn as nn

class NewsRNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_size=128,
        num_classes=4
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x, lengths):

        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        output, hidden = self.rnn(packed)

        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits

`nn.Embedding`
nn.Embedding(30000, 128)

turns:

word ID into a learned vector:

word 231
        ↓
[0.12, -0.41, ..., 0.67]

128 numbers.

These embedding values are learned during training.


`nn.RNN`

nn.RNN(
    input_size=128,
    hidden_size=128
)

means every timestep receives:

128-dimensional embedding

and maintains:

128-dimensional hidden state
batch_first=True

Makes input shape:

[batch, sequence, features]

For us:

[128, 80, 128]

rather than:

[80, 128, 128]

Much easier to understand.

pack_padded_sequence

This tells the RNN:

Don't waste computation treating PAD tokens as actual words.

Very useful for variable-length NLP.

`nn.Linear(128, 4)`

Takes the RNN's final representation:

128 values

and generates:

4 logits

One for each news category.

## Model

In [20]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NewsRNN(
    vocab_size=VOCAB_SIZE
).to(device)

print(model)

NewsRNN(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)


In [22]:
print(device)

cuda


## Loss + optimizer

In [23]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

## Training cycle

### Every batch:

1. Clear old gradients
2. Forward pass
3. Calculate loss
4. Backpropagation
5. Update weights

In [24]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for x, lengths, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predicted = logits.argmax(dim=1)

        correct += (predicted == y).sum().item()
        total += y.size(0)

    train_accuracy = correct / total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Accuracy: {train_accuracy:.4f}"
    )

Epoch 1/5 | Loss: 0.8223 | Accuracy: 0.6547
Epoch 2/5 | Loss: 0.5086 | Accuracy: 0.8309
Epoch 3/5 | Loss: 0.4320 | Accuracy: 0.8628
Epoch 4/5 | Loss: 0.4180 | Accuracy: 0.8671
Epoch 5/5 | Loss: 0.4096 | Accuracy: 0.8695


**### model.train()**

Activates training behaviour.

Particularly important when networks contain:

Dropout
BatchNorm



**### optimizer.zero_grad()**


PyTorch accumulates gradients.

Therefore old gradients must be removed before the next backward pass.

**### loss.backward()**

This is backpropagation.

PyTorch's Autograd calculates:

$$ \frac{\partial Loss}{\partial Weight} $$

for all trainable model weights.

### **optimizer.step()**

Actually updates:

weights
biases
embedding vectors
RNN parameters

using the gradients.

## Validation function

In [25]:
def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, lengths, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x, lengths)

            predictions = logits.argmax(dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    return correct / total

In [26]:
val_accuracy = evaluate(model, val_loader)

print(f"Validation accuracy: {val_accuracy:.4f}")

Validation accuracy: 0.8246


In [28]:
idx2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

def predict_news(text):
    model.eval()
    with torch.no_grad():
        ids, length = encode_text(text)
        input_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
        length_tensor = torch.tensor([length], dtype=torch.long).to(device)

        logits = model(input_tensor, length_tensor)
        prediction = logits.argmax(dim=1).item()

    return idx2label[prediction]

predict_news("Apple launches a new AI-powered chip")

'Sci/Tech'

In [29]:
predict_news("Messi retured from football")

'Sports'

In [30]:
predict_news("Messi retured from acting")

'Sci/Tech'

In [31]:
predict_news("Messi retured from study")

'Sci/Tech'

In [32]:
predict_news("Messi retured from market")

'Sci/Tech'